# Grad-CAM Implementation with Modern PyTorch

This notebook includes an updated Grad-CAM implementation using the latest PyTorch library. This replaces deprecated Keras code with PyTorch code for visualizing Convolutional Neural Networks (CNNs).

## Import Required Libraries

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

## Load Pre-trained Model

In [ ]:
model = models.resnet50(pretrained=True)
model.eval()

# Register hook to capture the gradients
class Hook:
    def __init__(self, module):
        self.gradients = None
        self.hook = module.register_backward_hook(self.save_gradient)

    def save_gradient(self, module, input, output):
        self.gradients = input[0].grad

# Select the target layer
target_layer = model.layer4[2]
model.eval()
hook = Hook(target_layer)

## Prepare Input Image

In [ ]:
image_path = 'path/to/your/image.jpg'
image = Image.open(image_path)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
input_tensor = preprocess(image)
input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch as expected by the model

## Forward and Backward Pass

In [ ]:
# Forward pass
output = model(input_batch)
# Perform backward pass
class_idx = output.argmax().item()
output[0, class_idx].backward()

# Get the weights and the activation from the hook
gradients = hook.gradients.data.numpy()[0]
activations = target_layer(input_batch).data.numpy()[0]

## Calculate Grad-CAM

In [ ]:
# Calculate the Grad-CAM weights
weights = np.mean(gradients, axis=(1, 2))
cam = np.zeros(activations.shape[1:], dtype=np.float32)
for i in range(len(weights)):
    cam += weights[i] * activations[i, :, :]

# Apply ReLU to the CAM
cam = np.maximum(cam, 0)
# Normalize the CAM to [0, 1]
cam = cam / np.max(cam)

## Overlay the CAM on the Image

In [ ]:
original_image = Image.open(image_path)
heatmap = np.uint8(255 * cam)
heatmap = np.array(Image.fromarray(heatmap).resize(original_image.size))
heatmap = np.float32(heatmap) / 255
superimposed_img = heatmap + np.array(original_image) / 255
superimposed_img = superimposed_img / np.max(superimposed_img)

# Display the original image and the Grad-CAM
plt.imshow(superimposed_img)
plt.axis('off')
plt.show()